# 市场状态分析 Notebook

这个 notebook 用来调用 `market_data.py` 和 `market_state_lib.py`，完成以下几类分析：

- 从 SQLite 数据库读取指定标的的历史数据
- 计算对数收益率、成交活跃度比值对数、Yang-Zhang 波动率的分布
- 计算 3×3 状态概率矩阵、9 状态概率向量
- 计算 9×9 / 4×4 状态转移概率矩阵
- 扫描不同时间粒度 `delta_t`，观察“微观均值回归”到“宏观动量延续”的临界点
- 可选地对多个标的做横向对比

建议使用顺序：

1. 先运行“环境与路径”“数据库连接与参数”两部分。
2. 再修改 `TS_CODE`、`START_DATE`、`END_DATE`、`price_threshold`、
`activity_threshold`、`LBW`、`DELTA_T`。
1. 依次运行后面的分析单元。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pathlib import Path
import sys
import warnings
from market_data import load_tushare_token, create_manager
from market_state_lib import (
    PHASE_ALL,
    load_prices_from_store,
    run_full_market_state_analysis,
)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings('ignore')


## 数据库连接与参数

下面这段保持了你当前能正常连接数据库的写法，只是多加了一点路径兼容。

- `TS_CODE`：单标的分析用的代码
- `COMPARE_TS_CODES`：多标的横向对比列表
- `ACTIVITY_COL`：默认用 `vol`，如果你后面有换手率列，可以改成对应列名
- `DELTA_T`：时间粒度，单位是交易日根数
- `PHASE`：可以设为 `0/1/2/...`，也可以设为 `PHASE_ALL`
- `LBW`：回看历史窗口长度
- `price_threshold`=对数收益率阈值,
- `activity_threshold`:成交量比值对数阈值,


In [ ]:
TUSHARE_TOKEN = load_tushare_token()
DB_PATH = "data/db/market_data.db"

manager = create_manager(
    tushare_token=TUSHARE_TOKEN,
    db_path=DB_PATH,
    default_start_date='20150101',
    default_exchange='SSE',
)


COMPARE_TS_CODES = [
    "510300.SH",  # 沪深300ETF (核心权益：大盘蓝筹)
    #"510500.SH",  # 中证500ETF (核心权益：中盘成长)
    #"510170.SH",  # 50ETF (核心权益：大宗商品股票集合)
    #"159915.SZ",  # 创业板ETF (核心权益：高新成长 - 纯被动)
    "511090.SH",  # 30年国债ETF (避险资产：超长债，对利率极度敏感)
    "511010.SH",  # 国债ETF (基础配置：中长期债)
    "518880.SH",  # 黄金ETF (抗通胀/避险：金属商品)
    "159981.SZ",  # 能源ETF (抗通胀：能源/电力)
    "159985.SZ",  # 豆粕ETF (抗通胀：农产品期货)
    "501018.SH",  # 南方原油LOF (抗通胀：国际原油价格)
    "515100.SH",  # 红利低波100ETF (抗通胀：红利股)
]
TS_CODE = COMPARE_TS_CODES[5]

START_DATE = '20200101'
END_DATE = None

ACTIVITY_COL = 'vol'           # 如果后续有换手率列，可改成比如 'turnover_rate'
price_threshold=1e-5            #对数收益率阈值,
activity_threshold=1e-3             #:成交量比值对数阈值,
LBW = 720                      # 回看窗口长度
DELTA_T = 1                    # 时间粒度：5 表示 5 个交易日聚合为一根 bar
PHASE = PHASE_ALL              # 也可以写成 0, 1, 2, ...
YZ_WINDOW = 20
ANNUALIZE_YZ = False

DELTA_T_GRID = [1, 2, 3,4, 5,6,7,8,9, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60,70,80,90,100,110, 120]

print('DB_PATH =', DB_PATH)
print('TS_CODE =', TS_CODE)
print('PHASE   =', PHASE)


In [ ]:
instruments = manager.store.get_instruments(listed_only=True)
display(instruments.head(10))
print('listed instruments:', len(instruments))
print('sample ts_codes:', manager.store.get_ts_codes(listed_only=True)[:20])


## 绘图与汇总辅助函数

这里尽量只用 `matplotlib`，避免额外依赖。热力图会把矩阵里的概率直接标出来，方便看主对角线与副对角线。

In [ ]:
def plot_series_distribution(values, title: str, bins: int = 50):
    s = pd.Series(values).dropna()
    if s.empty:
        print(f'{title}: 没有可用数据')
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(s, bins=bins)
    axes[0].set_title(title + ' - Histogram')
    axes[0].set_xlabel('value')
    axes[0].set_ylabel('count')
    axes[0].grid(alpha=0.3)

    axes[1].boxplot(s, vert=True)
    axes[1].set_title(title + ' - Boxplot')
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_heatmap(df: pd.DataFrame, title: str, cmap: str = 'viridis', vmin: float | None = 0.0, vmax: float | None = 1.0):
    mat = pd.DataFrame(df).copy()
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(mat.values, cmap=cmap, aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks(range(mat.shape[1]))
    ax.set_yticks(range(mat.shape[0]))
    ax.set_xticklabels(mat.columns, rotation=45, ha='right')
    ax.set_yticklabels(mat.index)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            value = mat.iloc[i, j]
            text = f'{value:.2f}' if pd.notna(value) else 'nan'
            ax.text(j, i, text, ha='center', va='center', fontsize=9, color='white' if value > 0.5 else 'black')

    plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


def extract_core_outputs(result, phase_mode):
    if phase_mode == PHASE_ALL:
        prob_3x3 = result['phase_avg_state_prob_3x3']
        prob_9 = result['phase_avg_state_prob_9']
        trans_9 = result['phase_avg_transition_9x9']
        trans_4 = result['phase_avg_transition_4x4']
        ret_values = result['log_return_distribution_all_phases']['value']
        act_values = result['log_activity_ratio_distribution_all_phases']['value']
        yz_values = result['yang_zhang_vol_distribution_all_phases']['value']
    else:
        state_res = result['state_analysis']
        prob_3x3 = state_res.state_prob_3x3
        prob_9 = state_res.state_prob_9
        trans_9 = state_res.transition_9x9
        trans_4 = state_res.transition_4x4
        ret_values = result['log_return_distribution'].values
        act_values = result['log_activity_ratio_distribution'].values
        yz_values = result['yang_zhang_vol_distribution'].values
    return prob_3x3, prob_9, trans_9, trans_4, pd.Series(ret_values), pd.Series(act_values), pd.Series(yz_values)


def summarize_transition_4x4(mat: pd.DataFrame) -> dict:
    arr = pd.DataFrame(mat).astype(float).to_numpy()
    diag_sum = float(np.trace(arr))
    anti_diag_sum = float(np.fliplr(arr).trace())
    n = arr.shape[0] if arr.size > 0 else 1
    return {
        'diag_sum_4x4': diag_sum,
        'diag_mean_4x4': diag_sum / n,
        'anti_diag_sum_4x4': anti_diag_sum,
        'anti_diag_mean_4x4': anti_diag_sum / n,
        'trend_minus_reversal_4x4': (diag_sum - anti_diag_sum) / n,
    }
def summarize_transition_9x9(trans_9: pd.DataFrame) -> dict:
    """
    基于现有 9x9 行归一化状态转移矩阵，提取若干适合横向比较的指标。

    状态顺序固定为：
        0  (+1,+1)
        1  (+1,0)
        2  (+1,-1)
        3  (0,+1)
        4  (0,0)
        5  (0,-1)
        6  (-1,+1)
        7  (-1,0)
        8  (-1,-1)
    """
    arr = trans_9.to_numpy(dtype=float)

    return {
        #都是基于9*9状态转移概率矩阵，矩阵采用全局归一化，
        #矩阵的行是昨日
        #列是今日
        """
        排列顺序：
        • 多头区间 (Price = +1)
            ○ 状态 1: $(+1, +1)$ 涨且放量 （主升/逼空）
            ○ 状态 2: $(+1, 0)$ 涨且平量 （常态上涨）
            ○ 状态 3: $(+1, -1)$ 涨且缩量 （惜售/买盘枯竭）
        • 震荡区间 (Price = 0)
            ○ 状态 4: $(0, +1)$ 平且放量 （多空激烈交火/分歧）
            ○ 状态 5: $(0, 0)$ 平且平量 （绝对死水）
            ○ 状态 6: $(0, -1)$ 平且缩量 （毫无交易意愿）
        • 空头区间 (Price = -1)
            ○ 状态 7: $(-1, +1)$ 跌且放量 （恐慌抛售/爆量抄底）
            ○ 状态 8: $(-1, 0)$ 跌且平量 （常态下跌）
            ○ 状态 9: $(-1, -1)$ 跌且缩量 （阴跌绵绵/流动性丧失
        """

        # 取和
        'diag_sum_9x9': float(np.trace(arr)),          # 主对角线：状态完全保持
        'pos_to_neg_sum_9x9': float(arr[0:3, 6:9].sum()),   # 右上角3x3：正收益 -> 负收益
        'neg_to_pos_sum_9x9': float(arr[6:9, 0:3].sum()),   # 左下角3x3：负收益 -> 正收益
        'pos_keep_sum_9x9': float(arr[0:3, 0:3].sum()),     # 左上角3x3：正收益保持
        'neg_keep_sum_9x9': float(arr[6:9, 6:9].sum()),     # 右下角3x3：负收益保持

        # 取元素值
        'state_00': float(arr[0, 0]),   # (+1,+1) -> (+1,+1)
        'state_88': float(arr[8, 8]),   # (-1,-1) -> (-1,-1)
        'state_08': float(arr[0, 8]),   # (+1,+1) -> (-1,-1)
        'state_80': float(arr[8, 0]),   # (-1,-1) -> (+1,+1)
        'state_62': float(arr[6, 2]),   # (-1,+1) -> (+1,-1)
        'state_26': float(arr[2, 6]),   # (+1,-1) -> (-1,+1)
    }

## 读取单个标的历史数据

In [ ]:
df = load_prices_from_store(
    manager.store,
    ts_code=TS_CODE,
    start_date=START_DATE,
    end_date=END_DATE,
)

df = df.sort_values('trade_date').reset_index(drop=True)
print('shape =', df.shape)
display(df.head())
display(df.tail())


In [ ]:
if df.empty:
    raise ValueError(f'{TS_CODE} 在当前日期区间内没有数据，请检查 ts_code / 日期范围 / 数据库路径。')

phase_mode = PHASE if DELTA_T > 1 else 0
result = run_full_market_state_analysis(
    df=df,
    price_threshold = price_threshold,
    activity_threshold=activity_threshold,
    lbw=LBW,
    delta_t=DELTA_T,
    phase=phase_mode,
    activity_col=ACTIVITY_COL,
    yz_window=YZ_WINDOW,
    annualize_yz=ANNUALIZE_YZ,
)

prob_3x3, prob_9, trans_9, trans_4, ret_values, act_values, yz_values = extract_core_outputs(result, phase_mode)

print('phase_mode =', phase_mode)
print('return obs  =', ret_values.dropna().shape[0])
print('activity obs=', act_values.dropna().shape[0])
print('yz obs      =', yz_values.dropna().shape[0])


## 分布分析

这里分别看：

- 对数收益率分布
- 活跃度比值对数分布
- Yang-Zhang 波动率分布

当 `PHASE = PHASE_ALL` 时，这里展示的是把所有相位结果合并后的总体分布。

In [ ]:
plot_series_distribution(ret_values, f'{TS_CODE} | log return | delta_t={DELTA_T} | phase={phase_mode}')
plot_series_distribution(act_values, f'{TS_CODE} | log {ACTIVITY_COL} ratio | delta_t={DELTA_T} | phase={phase_mode}')
plot_series_distribution(yz_values, f'{TS_CODE} | Yang-Zhang vol | delta_t={DELTA_T} | phase={phase_mode}')

summary_df = pd.DataFrame({
    'metric': ['log_return', f'log_{ACTIVITY_COL}_ratio', 'yang_zhang_vol'],
    'count': [ret_values.count(), act_values.count(), yz_values.count()],
    'mean': [ret_values.mean(), act_values.mean(), yz_values.mean()],
    'std': [ret_values.std(), act_values.std(), yz_values.std()],
    'p05': [ret_values.quantile(0.05), act_values.quantile(0.05), yz_values.quantile(0.05)],
    'p50': [ret_values.quantile(0.50), act_values.quantile(0.50), yz_values.quantile(0.50)],
    'p95': [ret_values.quantile(0.95), act_values.quantile(0.95), yz_values.quantile(0.95)],
})
display(summary_df)


## (对数收益率，对数成交量比值)二维分布

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 这里假设你前面已经有：
# state_frame
# PRICE_THRESHOLD
# ACTIVITY_THRESHOLD
sf = result["state_analysis"].state_frame
plot_df = sf[["log_return", "log_activity_ratio"]].dropna().copy()

# 点太多时抽样，避免图过于糊
max_points = 5000
if len(plot_df) > max_points:
    plot_df = plot_df.sample(max_points, random_state=42)

plt.figure(figsize=(8, 8))
plt.scatter(
    plot_df["log_return"],
    plot_df["log_activity_ratio"],
    s=12,
    alpha=0.35
)

# 中心线
plt.axvline(0, linestyle="--", linewidth=1)
plt.axhline(0, linestyle="--", linewidth=1)
  
# 阈值线
plt.axvline(price_threshold, linestyle=":", linewidth=1.2)
plt.axvline(-price_threshold, linestyle=":", linewidth=1.2)
plt.axhline(activity_threshold, linestyle=":", linewidth=1.2)
plt.axhline(-activity_threshold, linestyle=":", linewidth=1.2)

plt.xlabel("log_return")
plt.ylabel("log_activity_ratio")
plt.title("Scatter of log_return vs log_activity_ratio")
plt.grid(True, alpha=0.2)
plt.show()

## 状态概率矩阵与状态转移矩阵

解释：

- `3x3`：横轴是活跃度状态，纵轴是价格状态
- `9x9`：完整九状态转移矩阵
- `4x4`：只保留 `(±1, ±1)` 四种状态，更适合看趋势保持与反转
- 主对角线：状态保持
- 右上到左下副对角线：强反转

In [ ]:
display(prob_3x3)
display(prob_9.to_frame('probability'))

plot_heatmap(prob_3x3, f'{TS_CODE} | 3x3 state probability | delta_t={DELTA_T} | phase={phase_mode}')
plot_heatmap(trans_9, f'{TS_CODE} | 9x9 transition matrix | delta_t={DELTA_T} | phase={phase_mode}')
plot_heatmap(trans_4, f'{TS_CODE} | 4x4 transition matrix | delta_t={DELTA_T} | phase={phase_mode}')

transition_scores = pd.DataFrame([summarize_transition_4x4(trans_4)])
display(transition_scores)


## 扫描不同时间粒度 `delta_t`

这部分是最贴近你想法的分析之一：

- 固定 `LBW`、`price_threshold` 、`activity_threshold`
- 扫描不同 `delta_t`
- 观察 4 状态转移矩阵的主对角线均值与副对角线均值

一个粗略理解是：

- 主对角线更高：状态延续更强，偏动量
- 副对角线更高：状态强反转更强，偏均值回归

如果两者在某个时间尺度附近出现交叉，就可能对应你说的那个临界点。

In [ ]:
sweep_rows = []

for dt in DELTA_T_GRID:
    phase_mode_i = PHASE_ALL if dt > 1 else 0

    result_i = run_full_market_state_analysis(
        df=df,
        price_threshold=price_threshold,
        activity_threshold=activity_threshold,
        lbw=LBW,
        delta_t=dt,
        phase=phase_mode_i,
        activity_col=ACTIVITY_COL,
        yz_window=YZ_WINDOW,
        annualize_yz=ANNUALIZE_YZ,
    )

    prob_3x3_i, prob_9_i, trans_9_i, trans_4_i, ret_i, act_i, yz_i = extract_core_outputs(result_i, phase_mode_i)

    scores_i = summarize_transition_9x9(trans_9_i)

    sweep_rows.append({
        'delta_t': dt,
        'phase_mode': str(phase_mode_i),
        'return_std': ret_i.std(),
        'activity_std': act_i.std(),
        'yz_mean': yz_i.mean(),

        'state_(0,0)': float(prob_9_i.get('(0,0)', np.nan)),
        'state_(+1,+1)': float(prob_9_i.get('(+1,+1)', np.nan)),
        'state_(-1,-1)': float(prob_9_i.get('(-1,-1)', np.nan)),

        **scores_i,
    })

sweep_df = pd.DataFrame(sweep_rows).sort_values('delta_t').reset_index(drop=True)
display(sweep_df)

In [ ]:
if not sweep_df.empty:
    def plot_sweep_line(df, y_col, title, ylabel):
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(df['delta_t'], df[y_col], marker='o')
        ax.set_title(title)
        ax.set_xlabel('delta_t')
        ax.set_ylabel(ylabel)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

    # =========================
    # 一、整体价格延续 / 反转
    # =========================

    sweep_df = sweep_df.copy()
    sweep_df['price_reversal_sum_9x9'] = (
        sweep_df['pos_to_neg_sum_9x9'] + sweep_df['neg_to_pos_sum_9x9']
    )
    sweep_df['price_keep_sum_9x9'] = (
        sweep_df['pos_keep_sum_9x9'] + sweep_df['neg_keep_sum_9x9']
    )
    sweep_df['price_keep_minus_reversal_9x9'] = (
        sweep_df['price_keep_sum_9x9'] - sweep_df['price_reversal_sum_9x9']
    )

    plot_sweep_line(
        sweep_df,
        y_col='price_keep_sum_9x9',
        title='price_keep_sum_9x9 vs delta_t',
        ylabel='price_keep_sum_9x9',
    )

    plot_sweep_line(
        sweep_df,
        y_col='price_reversal_sum_9x9',
        title='price_reversal_sum_9x9 vs delta_t',
        ylabel='price_reversal_sum_9x9',
    )

    plot_sweep_line(
        sweep_df,
        y_col='price_keep_minus_reversal_9x9',
        title='price_keep_minus_reversal_9x9 vs delta_t',
        ylabel='price_keep_sum_9x9 - price_reversal_sum_9x9',
    )

    # =========================
    # 二、块级别结构
    # =========================

    plot_sweep_line(
        sweep_df,
        y_col='diag_sum_9x9',
        title='diag_sum_9x9 vs delta_t',
        ylabel='diag_sum_9x9',
    )

    plot_sweep_line(
        sweep_df,
        y_col='pos_keep_sum_9x9',
        title='pos_keep_sum_9x9 vs delta_t',
        ylabel='pos_keep_sum_9x9',
    )

    plot_sweep_line(
        sweep_df,
        y_col='neg_keep_sum_9x9',
        title='neg_keep_sum_9x9 vs delta_t',
        ylabel='neg_keep_sum_9x9',
    )

    plot_sweep_line(
        sweep_df,
        y_col='pos_to_neg_sum_9x9',
        title='pos_to_neg_sum_9x9 vs delta_t',
        ylabel='pos_to_neg_sum_9x9',
    )

    plot_sweep_line(
        sweep_df,
        y_col='neg_to_pos_sum_9x9',
        title='neg_to_pos_sum_9x9 vs delta_t',
        ylabel='neg_to_pos_sum_9x9',
    )

    # =========================
    # 三、关键路径元素
    # =========================

    plot_sweep_line(
        sweep_df,
        y_col='state_00',
        title='state_00 vs delta_t',
        ylabel='(+1,+1) -> (+1,+1)',
    )

    plot_sweep_line(
        sweep_df,
        y_col='state_88',
        title='state_88 vs delta_t',
        ylabel='(-1,-1) -> (-1,-1)',
    )

    plot_sweep_line(
        sweep_df,
        y_col='state_08',
        title='state_08 vs delta_t',
        ylabel='(+1,+1) -> (-1,-1)',
    )

    plot_sweep_line(
        sweep_df,
        y_col='state_80',
        title='state_80 vs delta_t',
        ylabel='(-1,-1) -> (+1,+1)',
    )

    plot_sweep_line(
        sweep_df,
        y_col='state_62',
        title='state_62 vs delta_t',
        ylabel='(-1,+1) -> (+1,-1)',
    )

    plot_sweep_line(
        sweep_df,
        y_col='state_26',
        title='state_26 vs delta_t',
        ylabel='(+1,-1) -> (-1,+1)',
    )

## 多标的横向对比

这里用同一组参数，对多个标的的 4 状态转移特征做横向比较。适合回答你说的“空间上对比：不同标的的性质”。

In [ ]:
compare_rows = []

for code in COMPARE_TS_CODES:
    df_i = load_prices_from_store(
        manager.store,
        ts_code=code,
        start_date=START_DATE,
        end_date=END_DATE,
    ).sort_values('trade_date').reset_index(drop=True)

    if df_i.empty:
        print(f'skip {code}: no data')
        continue

    phase_mode_i = PHASE if DELTA_T > 1 else 0

    result_i = run_full_market_state_analysis(
        df=df_i,
        price_threshold=price_threshold,
        activity_threshold=activity_threshold,
        lbw=LBW,
        delta_t=DELTA_T,
        phase=phase_mode_i,
        activity_col=ACTIVITY_COL,
        yz_window=YZ_WINDOW,
        annualize_yz=ANNUALIZE_YZ,
    )

    prob_3x3_i, prob_9_i, trans_9_i, trans_4_i, ret_i, act_i, yz_i = extract_core_outputs(result_i, phase_mode_i)

    scores_i = summarize_transition_9x9(trans_9_i)

    # 可选：再补几个更直观的汇总指标
    price_reversal_sum_9x9 = scores_i['pos_to_neg_sum_9x9'] + scores_i['neg_to_pos_sum_9x9']
    price_keep_sum_9x9 = scores_i['pos_keep_sum_9x9'] + scores_i['neg_keep_sum_9x9']
    price_keep_minus_reversal_9x9 = price_keep_sum_9x9 - price_reversal_sum_9x9

    compare_rows.append({
        'ts_code': code,
        'n_bars_raw': len(df_i),
        'return_std': ret_i.std(),
        'yz_mean': yz_i.mean(),

        'state_(+1,+1)': float(prob_9_i.get('(+1,+1)', np.nan)),
        'state_(0,0)': float(prob_9_i.get('(0,0)', np.nan)),
        'state_(-1,-1)': float(prob_9_i.get('(-1,-1)', np.nan)),

        'price_reversal_sum_9x9': price_reversal_sum_9x9,
        'price_keep_sum_9x9': price_keep_sum_9x9,
        'price_keep_minus_reversal_9x9': price_keep_minus_reversal_9x9,

        **scores_i,
    })

compare_df = (
    pd.DataFrame(compare_rows)
    .sort_values('price_keep_minus_reversal_9x9', ascending=False)
    .reset_index(drop=True)
)

display(compare_df)

In [ ]:
if not compare_df.empty:
    def plot_compare_bar(df, col, title, ylabel, sort_by_col=None):
        plot_df = df.copy()
        if sort_by_col is not None:
            plot_df = plot_df.sort_values(sort_by_col, ascending=False).reset_index(drop=True)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(plot_df['ts_code'], plot_df[col])
        ax.axhline(0, linestyle='--', alpha=0.5)
        ax.set_title(title)
        ax.set_xlabel('ts_code')
        ax.set_ylabel(ylabel)
        ax.grid(axis='y', alpha=0.3)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

    # =========================
    # 一、整体汇总指标
    # =========================

    plot_compare_bar(
        compare_df,
        col='price_keep_sum_9x9',
        title=f'price_keep_sum_9x9 of different targets under delta_t={DELTA_T}',
        ylabel='price_keep_sum_9x9',
        sort_by_col='price_keep_sum_9x9',
    )

    plot_compare_bar(
        compare_df,
        col='price_reversal_sum_9x9',
        title=f'price_reversal_sum_9x9 of different targets under delta_t={DELTA_T}',
        ylabel='price_reversal_sum_9x9',
        sort_by_col='price_reversal_sum_9x9',
    )

    plot_compare_bar(
        compare_df,
        col='price_keep_minus_reversal_9x9',
        title=f'price_keep_minus_reversal_9x9 of different targets under delta_t={DELTA_T}',
        ylabel='price_keep_sum_9x9 - price_reversal_sum_9x9',
        sort_by_col='price_keep_minus_reversal_9x9',
    )

    plot_compare_bar(
        compare_df,
        col='diag_sum_9x9',
        title=f'diag_sum_9x9 of different targets under delta_t={DELTA_T}',
        ylabel='diag_sum_9x9',
        sort_by_col='diag_sum_9x9',
    )

    plot_compare_bar(
        compare_df,
        col='pos_keep_sum_9x9',
        title=f'pos_keep_sum_9x9 of different targets under delta_t={DELTA_T}',
        ylabel='pos_keep_sum_9x9',
        sort_by_col='pos_keep_sum_9x9',
    )

    plot_compare_bar(
        compare_df,
        col='neg_keep_sum_9x9',
        title=f'neg_keep_sum_9x9 of different targets under delta_t={DELTA_T}',
        ylabel='neg_keep_sum_9x9',
        sort_by_col='neg_keep_sum_9x9',
    )

    plot_compare_bar(
        compare_df,
        col='pos_to_neg_sum_9x9',
        title=f'pos_to_neg_sum_9x9 of different targets under delta_t={DELTA_T}',
        ylabel='pos_to_neg_sum_9x9',
        sort_by_col='pos_to_neg_sum_9x9',
    )

    plot_compare_bar(
        compare_df,
        col='neg_to_pos_sum_9x9',
        title=f'neg_to_pos_sum_9x9 of different targets under delta_t={DELTA_T}',
        ylabel='neg_to_pos_sum_9x9',
        sort_by_col='neg_to_pos_sum_9x9',
    )

    # =========================
    # 二、关键元素路径
    # =========================

    plot_compare_bar(
        compare_df,
        col='state_00',
        title=f'state_00 of different targets under delta_t={DELTA_T}',
        ylabel='(+1,+1) -> (+1,+1)',
        sort_by_col='state_00',
    )

    plot_compare_bar(
        compare_df,
        col='state_88',
        title=f'state_88 of different targets under delta_t={DELTA_T}',
        ylabel='(-1,-1) -> (-1,-1)',
        sort_by_col='state_88',
    )

    plot_compare_bar(
        compare_df,
        col='state_08',
        title=f'state_08 of different targets under delta_t={DELTA_T}',
        ylabel='(+1,+1) -> (-1,-1)',
        sort_by_col='state_08',
    )

    plot_compare_bar(
        compare_df,
        col='state_80',
        title=f'state_80 of different targets under delta_t={DELTA_T}',
        ylabel='(-1,-1) -> (+1,+1)',
        sort_by_col='state_80',
    )

    plot_compare_bar(
        compare_df,
        col='state_62',
        title=f'state_62 of different targets under delta_t={DELTA_T}',
        ylabel='(-1,+1) -> (+1,-1)',
        sort_by_col='state_62',
    )

    plot_compare_bar(
        compare_df,
        col='state_26',
        title=f'state_26 of different targets under delta_t={DELTA_T}',
        ylabel='(+1,-1) -> (-1,+1)',
        sort_by_col='state_26',
    )

## 多标的在不同时间颗粒度下的分析


In [ ]:
# =========================
# 多标的 + 时间颗粒度 扫描
# 同一个指标一张图，多个标的一起画，横轴是 delta_t
# =========================

asset_dt_rows = []

for code in COMPARE_TS_CODES:
    df_i = load_prices_from_store(
        manager.store,
        ts_code=code,
        start_date=START_DATE,
        end_date=END_DATE,
    ).sort_values('trade_date').reset_index(drop=True)

    if df_i.empty:
        print(f'skip {code}: no data')
        continue

    for dt in DELTA_T_GRID:
        phase_mode_i = PHASE_ALL if dt > 1 else 0

        result_i = run_full_market_state_analysis(
            df=df_i,
            price_threshold=price_threshold,
            activity_threshold=activity_threshold,
            lbw=LBW,
            delta_t=dt,
            phase=phase_mode_i,
            activity_col=ACTIVITY_COL,
            yz_window=YZ_WINDOW,
            annualize_yz=ANNUALIZE_YZ,
        )

        prob_3x3_i, prob_9_i, trans_9_i, trans_4_i, ret_i, act_i, yz_i = extract_core_outputs(result_i, phase_mode_i)
        scores_i = summarize_transition_9x9(trans_9_i)

        price_reversal_sum_9x9 = scores_i['pos_to_neg_sum_9x9'] + scores_i['neg_to_pos_sum_9x9']
        price_keep_sum_9x9 = scores_i['pos_keep_sum_9x9'] + scores_i['neg_keep_sum_9x9']
        price_keep_minus_reversal_9x9 = price_keep_sum_9x9 - price_reversal_sum_9x9

        asset_dt_rows.append({
            'ts_code': code,
            'delta_t': dt,
            'phase_mode': str(phase_mode_i),

            'return_std': ret_i.std(),
            'activity_std': act_i.std(),
            'yz_mean': yz_i.mean(),

            'state_(+1,+1)': float(prob_9_i.get('(+1,+1)', np.nan)),
            'state_(0,0)': float(prob_9_i.get('(0,0)', np.nan)),
            'state_(-1,-1)': float(prob_9_i.get('(-1,-1)', np.nan)),

            'price_reversal_sum_9x9': price_reversal_sum_9x9,
            'price_keep_sum_9x9': price_keep_sum_9x9,
            'price_keep_minus_reversal_9x9': price_keep_minus_reversal_9x9,

            **scores_i,
        })

asset_dt_df = (
    pd.DataFrame(asset_dt_rows)
    .sort_values(['ts_code', 'delta_t'])
    .reset_index(drop=True)
)

display(asset_dt_df)


# =========================
# 绘图函数
# 同一个指标一张图，多个标的一起画
# =========================

def plot_metric_vs_delta_t_multi_asset(df, metric_col, title=None, ylabel=None):
    plot_df = df[['ts_code', 'delta_t', metric_col]].dropna().copy()
    if plot_df.empty:
        print(f'{metric_col}: no valid data')
        return

    fig, ax = plt.subplots(figsize=(10, 6))

    for code, g in plot_df.groupby('ts_code', sort=False):
        g = g.sort_values('delta_t')
        ax.plot(
            g['delta_t'],
            g[metric_col],
            marker='o',
            linewidth=1.8,
            label=code,
        )

    ax.set_title(title if title is not None else f'{metric_col} vs delta_t')
    ax.set_xlabel('delta_t')
    ax.set_ylabel(ylabel if ylabel is not None else metric_col)
    ax.grid(True, alpha=0.3)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


# =========================
# 你最关心的几个 9x9 指标
# 可自行增删
# =========================

METRICS_TO_PLOT = [
    # 总体延续 / 反转
    'price_keep_sum_9x9',
    'price_reversal_sum_9x9',
    'price_keep_minus_reversal_9x9',

    # 块级别
    'diag_sum_9x9',
    'pos_keep_sum_9x9',
    'neg_keep_sum_9x9',
    'pos_to_neg_sum_9x9',
    'neg_to_pos_sum_9x9',

    # 关键路径
    'state_00',
    'state_88',
    'state_08',
    'state_80',
    'state_62',
    'state_26',
]

for metric in METRICS_TO_PLOT:
    plot_metric_vs_delta_t_multi_asset(
        asset_dt_df,
        metric_col=metric,
        title=f'{metric} vs delta_t for multiple assets',
        ylabel=metric,
    )

## 可选：导出结果到 CSV

这个单元会把当前单标的分析中最常用的几个结果导出到本地，方便后续在别的脚本里继续分析或制图。

In [ ]:
OUTPUT_DIR = OUTPUT_DIR = Path(f"analysis_outputs/{TS_CODE.replace('.', '_')}_dt{DELTA_T}_lbw{LBW}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prob_3x3.to_csv(OUTPUT_DIR / 'state_prob_3x3.csv', encoding='utf-8-sig')
prob_9.to_frame('probability').to_csv(OUTPUT_DIR / 'state_prob_9.csv', encoding='utf-8-sig')
trans_9.to_csv(OUTPUT_DIR / 'transition_9x9.csv', encoding='utf-8-sig')
trans_4.to_csv(OUTPUT_DIR / 'transition_4x4.csv', encoding='utf-8-sig')
summary_df.to_csv(OUTPUT_DIR / 'distribution_summary.csv', index=False, encoding='utf-8-sig')
sweep_df.to_csv(OUTPUT_DIR / 'delta_t_sweep.csv', index=False, encoding='utf-8-sig')

print('saved to:', OUTPUT_DIR)
